This notebook will run the code for analysis of BD Rhapsody human BCR files for VDJ analysis. 
Python version 3.12.9
Date started: 04-Jan-2026
Author: Amelia Fisher
Based on: https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.io.read_airr.html#scirpy.io.read_airr, https://scirpy.scverse.org/en/latest/tutorials/tutorial_5k_bcr.html
 
Run on: Aire High Performance Computer, University of Leeds, UK

Input requirements:

1) VDJ_Dominant_Contigs_AIRR file from BD Rhapsody Valsera (formerly SevenBridges) primary analysis pipeline. Alternative methods to generate AIRR files and AIRR file definition can be found elsewhere and are beyond the scope of this notebook. 

2) Note that the AIRR file will contain all reads from a multiplexed batch if multiplexing by sample tag (SMK) has been used on the BD Rhapsody platform. If individual sample files are required, these will need to be generated individually (using a for loop to separate out tags per cell_id or similar) prior to use of this pipeline.

3) No batch correction is required for the dataset this was designed for. This is because the dominant BCR contig is either present or absent, it does not have a relative expression value like standard RNA seq. Check to see if this assumption applies to your data too. 

4) H5mu file of transcriptomic data which can be edited and generated in BDRhapsody cellismo before exporting as this file format and being used here. If I want to use targeted panel transcriptome information, it will need to have batch correction / normalisation and other preprocessing steps done in scanpy - this is embedded in the BCR_removeCLL_clonalAndBCRtranscriptomic.ipynb. Here the transcriptomic mdata file is for annotation and BCR purposes and is done in cellismo so BCR and AbSeq annotations are consistent across the dataset. Transcriptome data from the targeted panel (that requires batch correction and normalisation for DE) is not used here. 

Checks - checks for dataframe sizes, columns and a basline list of fields for BD Rhapsody AIRR has been put in here at useful points. These can be removed if not required. This information should assist in changing the arguments of code as required throughout the complex multi-dimensional dataset. 

Not working: 
1) Repertoire overlap plot does not save. 
2) I can't get the data from the violin plot to any sort of format that would output me a useful p-value. 


## Set-up ##

In [ ]:
# Import python modules

import numpy as np
import pandas as pandas
import scanpy as sc
import scirpy as ir 
import muon as mu
import skbio as sk
import os as os
from muon import MuData
from matplotlib import pyplot as plt, cm as mpl_cm
from cycler import cycler

#sc.set_figure_params(figsize=(6, 6))  ## generates an error, get this code for current versions if its needed.
sc.settings.verbosity = 2  # verbosity: errors (0), warnings (1), info (2), hints (3)

In [ ]:
sc.logging.print_header()

# Set-up paths etc

In [ ]:
import os

# change paths throught notebook

# Batch 1 data
#multi_sample = False
#work_folder = "PATH/runFolders/Batch1_test"
#h5mu_file = "Batch1_usedForTests_muon_v2"
#airr_file = "Batch1_VDJ_Dominant_Contigs_AIRR.tsv"

# Note that batch5 had a processing error at the library prep stage, which resulted in very low quantity of the BCR amplifying. This had lead to a lot of unassigned BCR CDR3 nt sequences in this dataset.

# Batch 1 TO 8
multi_sample = True
VAR = "mdata:Sample_Tag_Name"
work_folder = "PATH/runFolders/AllData"
out_folder = os.path.join("PATH/outputs", "Sample_Tag_Name_v2")
h5mu_file = "AllData_rawAnno.h5mu"
airr_file = "Batch1_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_2 = "Batch2_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_3 = "Batch3_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_4 = "Batch4_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_5 = "Batch5_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_6 = "Batch6_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_7 = "Batch7_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_8 = "Batch8_VDJ_Dominant_Contigs_AIRR.tsv"


In [ ]:
print (out_folder)

## Get the data in ##

Get the data in, tutorial available at: https://scverse.org/scirpy/tags/v0.11.0/tutorials/tutorial_io.html#importing-data

Use h5mu annotations and BCR information and AIRR BCR data in this analysis. Transcriptomic data 

In [ ]:
# Get the transcriptomic data in, which has the group categories. 
# If I need different groupings I can make these in cellismo, and export as scanpy h5ad, and read as this file type instead. 
# AnnData and MuData options given here. 

#adata = ir.io.read_h5ad("PATH/Data/Testfiles/ipynb_test_data/Batch1_Test")

#adata.shape

#### MAKE SURE THE CORRECT FILE IS BEING CALLED HERE ####

mdata_path = os.path.join(work_folder, h5mu_file)

mdata = ir.io.read_h5mu(mdata_path)


mdata.shape

In [ ]:
# Multi-sample if there is different notatation between columns

mdata.obs.rename(columns={"Sample_Name":"Sample_Tag_Name"})

# Check my data object columns

mdata.obs.columns

# Filter out Multiplets and Undetermined first

This should be run for Cellismo processed results to remove any multiplet and undetermined categories that have left an imprint (i.e. empty columns etc) on the dataset, even if they have had the multiplets and undteremined cells removed in Cellismo. 

Doublet detection is not run because scrublet by scanpy detects doublets on a single sample by differences in cell type transcriptome. This does not apply to this dataset. Use the automatically detected multiple removal (based on sample tags) instead.



In [ ]:
mask = ~mdata.obs["Sample_Tag_Name"].isin(["Multiplet", 
                                           "Undetermined", 
                                           "<unassigned>"])
mdata_filtered = mdata[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in mdata_filtered.obs.columns:
    if pd.api.types.is_categorical_dtype(mdata_filtered.obs[col].dtype):
        mdata_filtered.obs[col] = mdata_filtered.obs[col].cat.remove_unused_categories()

mdata = mdata_filtered
mdata.shape

In [ ]:
# Check Cellismo cell calling annotation for the cell types identified and add any extras to the list here for filtering.

mask = ~mdata.obs["Cell_Type_Experimental"].isin(["Dendritic", 
                                           "Monocyte_classical",
                                           "Monocyte_nonclassical",
                                           "Natural_killer"
                                           "T_CD4_memory",
                                           "T_CD4_naive",
                                           "T_CD8_memory",
                                           "T_CD8_naive",
                                           "T_gamma_delta", 
                                           "<unassigned>"])
mdata_filtered = mdata[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in mdata_filtered.obs.columns:
    if pd.api.types.is_categorical_dtype(mdata_filtered.obs[col].dtype):
        mdata_filtered.obs[col] = mdata_filtered.obs[col].cat.remove_unused_categories()

mdata = mdata_filtered
mdata.shape

In [ ]:
# Check that all the multiplets and undetermined have been removed from all datasets, but filtering on a simple annotation. 
# After this mdata.shape should be the same size as the previous cell. If it is, filtering of multiplets and undetermined across the dataset has been successful. 

mask = ~mdata.obs["Sex"].isin(["multiplet", "undetermined", "<unassigned>"])
mdata_filtered = mdata[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in mdata_filtered.obs.columns:
    if pd.api.types.is_categorical_dtype(mdata_filtered.obs[col].dtype):
        mdata_filtered.obs[col] = mdata_filtered.obs[col].cat.remove_unused_categories()

mdata = mdata_filtered
mdata.shape

In [ ]:
# Filter out BCRs that are unassigned to a nt sequence OR filter out specific nucleotide BCR sequences as per output of BCR_multibatch_identifyCLLclone.ipynb
# Can list >1 BCR nt sequence using same method as in cell above.  
# Change BCRs listed

mask = ~mdata.obs["BCR_Heavy_CDR3_Junction_Nucleotide_Dominant"].isin(["<unassigned>"],
# "TGTGCGAAACATCTTCGATATAGCGGCTCGTCAGGAGATGCTTTTGATATCTGG",
# "TGTGCGAGAGTGTACTATGATAGTAGTGGTTATTACCTTGACTACTGA",
# "TGTGCGAGAGAAAGGGACCAACTGGCAGAGCACTGG",
# "TGTGGAGCAGCAACTGGTCTCTTCGTTGACTACTGG",
# "TGTGCGAAAGATGAGAGTGAGAGTGGGAGCTCAACCACTTCCCTTGACTACTGG",
# "TGTTCAAGAGATATAGTCCGGTCAGGTGGTCCCGCCTACTTTGACGACTGG",
# "TGTGCGAGAATTTATAGTGGGAGCTACTACTACTACTACTACGGTATGGACGTCTGG",
# "TGTGCGCGATGGAGTAAATATTACTGGTACTTCGATCTCTGG",
# "TGTGTGAGAGATGCTATCTGCTATACGAAATGTGTCCACTACTACTACATGGACGTCTGG",
# "TGTGCACGGTCCCGTCCCACCAATTACGCTTTTTGGAATGGTTATCTTAGGGGCGATGCTTTTGATATCTGG",
# "TGTGTACGGGCCCTTCATTACTATGGTTCGGGGAAAGGTACTGATGATGCTTTTGATATCTGG",
# "TGTGCGAAAGATATGTCCCTCTGGTGGCAGTGGCTTATTGACTACTACTACTACTACATGGACGTCTGG",
# "TGTGCGAGGGAAGGAACTACGGTGACTTACTTTGAATACTGG",
# "TGTGCGAAAGGTTGGATGAACTACTACTACGGTATGGACGTCTGG",
# "TGTGCGAGAGGCAGAGGGGATTACTATGATAGTAGTGGTTATTTATACTACTACTACGGTATGGACGTCTGG",
# "TGTGCGAGAGGGACTAGTGGGAGCTACGTCGACTACTGG",
# "TGTGCGAGCGACTTCCGATATTTTGACTGGTTACCCAACCCGGCCTACAACTGGTTCGACCCCTGG",
# "TGTGCGAGTGATCGCAACGGTATGGACGTCTGG",
# "TGTGCGAGAGATGCTAACGCTATGGACGTCTGG",
# "TGTGCGAAGGCAGTAGTGGCTGGTACCGGGTTTGTCGGCTACTTTGACCACTGG",
# "TGTGCGAGACGCCGCCCCAACGATTACGATTTTTGGAGTGGTTATTGGACCCCCGACTACTGG",
# "TGTACGAGAGGACACTACGGTTTGGACGTCTGG",
# "TGTACCGCAGAGTCCACTATCTGGTACCTCGGGAAAAAGTACTACTTTGACTACTGG",
# "TGTGCGAGAGATGGCCCCTATAGTAGGGATTATGATGCTTTTGATCTCTGG"]
)
mdata_filtered = mdata[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in mdata_filtered.obs.columns:
    if pd.api.types.is_categorical_dtype(mdata_filtered.obs[col].dtype):
        mdata_filtered.obs[col] = mdata_filtered.obs[col].cat.remove_unused_categories()

mdata = mdata_filtered
mdata.shape

In [ ]:
# Output this to have a look at where the cleaning affected the cells. Is it BCR removal or poor BCR reads in controls?

mdata.write("CleanedAllData.h5mu")

In [ ]:
# Check my mdata object columns

mdata.obs.head()

In [ ]:
# Get the AIRR file in with Anndata
# AnnData objects to merge AIRR with MuData downstream.
# The most recent BD Rhapsody primary anaylsis pipeline on Velsera (Seven Bridges) puts it into a standard AIRR format, so read_airr is used NOT read_bd_rhapsody!
# Defaults used for read_airr as per documentation https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.io.read_airr.html
# Dominant chains used in this analysis, path to dominant chain AIRR file. Unfiltered chains not needed at this stage.
# If I am importing more than one AnnData object, call each something different e.g. adata_1, adata_2 etc.
# If I need different groupings I can make these in cellismo, and export as scanpy h5ad, and read as this file type instead.

##################################################################
# -----------------PRE-PROCESS AIRR FILES-------------------------#
# -----------------CHANGE BATCH NUMBERS---------------------------#
##################################################################

if multi_sample:
    print("Processing multiple samples AIRR files before merging into MuData object.")
    import pandas

    batch1_airr = pandas.read_csv(os.path.join(work_folder, airr_file), sep="\t")
    batch1_airr["cell_id"] = batch1_airr["cell_id"].astype(str) + "_Batch1"
    batch2_airr = pandas.read_csv(os.path.join(work_folder, airr_file_2), sep="\t")
    batch2_airr["cell_id"] = batch2_airr["cell_id"].astype(str) + "_Batch2"
    batch3_airr = pandas.read_csv(os.path.join(work_folder, airr_file_3), sep="\t")
    batch3_airr["cell_id"] = batch3_airr["cell_id"].astype(str) + "_Batch3"
    batch4_airr = pandas.read_csv(os.path.join(work_folder, airr_file_4), sep="\t")
    batch4_airr["cell_id"] = batch4_airr["cell_id"].astype(str) + "_Batch4"
    batch5_airr = pandas.read_csv(os.path.join(work_folder, airr_file_5), sep="\t")
    batch5_airr["cell_id"] = batch5_airr["cell_id"].astype(str) + "_Batch5"
    batch6_airr = pandas.read_csv(os.path.join(work_folder, airr_file_6), sep="\t")
    batch6_airr["cell_id"] = batch6_airr["cell_id"].astype(str) + "_Batch6"
    batch7_airr = pandas.read_csv(os.path.join(work_folder, airr_file_7), sep="\t")
    batch7_airr["cell_id"] = batch7_airr["cell_id"].astype(str) + "_Batch7"
    batch8_airr = pandas.read_csv(os.path.join(work_folder, airr_file_8), sep="\t")
    batch8_airr["cell_id"] = batch8_airr["cell_id"].astype(str) + "_Batch8"

    adata_ir = ir.io.read_airr([pandas.concat([batch1_airr, 
                                             batch2_airr,
                                             batch3_airr,
                                             batch4_airr,
                                             batch5_airr,
                                             batch6_airr,
                                             batch7_airr,
                                             batch8_airr])
                                             ])
else:
    print("Processing single sample AIRR file before merging into MuData object.")
    adata_ir = ir.io.read_airr(os.path.join(work_folder, airr_file))

adata_ir.shape
# shape for BCR or TCR data should be (n, 0)

In [ ]:
# Check my adata_ir object columns

adata_ir.obs.head()

In [ ]:
# The AIRR rearrangement standard defines a set of fields to describe single receptor chains. One cell can have multiple receptor chains. This relationship is represented as an awkward array stored in adata.obsm["airr"].
adata_ir.obsm["airr"]

In [ ]:
adata_ir.obs.columns

## AIRR file from BD Rhaposdy fields available:

cell_id	cell_type_experimental
high_quality_cell_tcr_bcr	
locus	
sequence_id	
consensus_count	
umi_count	
sequence	
sequence_length	
sequence_aa	
sequence_aa_length	
sequence_alignment	
sequence_alignment_length	
sequence_alignment_aa	
sequence_alignment_aa_length	
germline_alignment	
germline_alignment_aa	
v_germline_alignment	
v_germline_alignment_aa	
d_germline_alignment	
d_germline_alignment_aa	
j_germline_alignment	
j_germline_alignment_aa	
v_germline_start	
v_germline_end	
d_germline_start	
d_germline_end	
j_germline_start	
j_germline_end	
np1_length	
np2_length	
junction	
junction_aa	
junction_anchored_aa	
productive	
rev_comp	
complete_vdj	
dominant	
putative_cell	
v_call	
v_support	
v_cigar	
v_sequence_start	
v_sequence_end	
d_call	
d_support	
d_cigar	
d_sequence_start	
d_sequence_end	
j_call	
j_support	
j_cigar	
j_sequence_start	
j_sequence_end	
c_call	
fwr1	
fwr1_aa	
fwr2	
fwr2_aa	
fwr3	
fwr3_aa	
fwr4	
fwr4_aa	
cdr1	
cdr1_aa	
cdr2	
cdr2_aa	
cdr3	
cdr3_aa

## Preprocessing of data and BCR quality control ##

https://scirpy.scverse.org/en/latest/api.html

In [ ]:
# Print data information - mdata (transcriptomic)

mdata

In [ ]:
# Print data information - adata (AIRR)

adata_ir

In [ ]:
# Quality control. Can get a lot of this info out of Cellismo too. 
# No filters applied here because CLL has atypical and unproductive sequences that may be clonally relevant.
# Selects primary and secondary chains according to the IR model. 

ir.pp.index_chains(adata_ir)
ir.tl.chain_qc(adata_ir)


## Merge transcriptomic and BCR AIRR data

In [ ]:
# ir.pp.merge_with_ir(mdata, adata_ir) - this is depreciated
# Use MuData

mdata_airr = MuData({"mdata": mdata, "airr": adata_ir})

In [ ]:
# Check new mdata_airr object
mdata_airr.obs.head

In [ ]:
# List column names of mdata_airr grouping categories --> add more here

#mdata.obs["GroupA_arms"].value_counts()
#mdata.obs["Sample_Tag_Name"].value_counts()

In [ ]:
# List modality keys of mdata_airr - this is not generating connectivity keys, which is an issue downstream. 

list(mdata_airr.mod.keys())

In [ ]:
# Check the variable for investigation - this is in the top part of the scripot where the directories are set.

print(VAR)

# Get some stats
Not likey to be that useful, but if need to evaluate method these could be needed. Only Sample_Tag_Name has been used here, because the rest will not be representative of the dataset given it's downsize based on a complete BCR sequence. 

In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# Batch 5 has an error of BCR library preparation: 51, 68, 71
# Batch 7 samples 76 and 80 were spoiled during processing. 

#ir.tl.group_abundance(mdata_airr, groupby= "mdata:Sample_Tag_Name")

ir.tl.group_abundance(mdata_airr, groupby= VAR)

In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# False = doesn't have an IR.  
# In controls these should all ne 'NA' and it will therefore give the number of cells with and without an IR.

#SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
 #                     groupby= "mdata:Sample_Tag_Name")

FILE = os.path.join(out_folder, "SampleAbundance.csv")

SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
                      groupby= VAR,
)

SampleAbundance_data_frame.to_csv(path_or_buf= FILE)

Next 3 cells are for BCell_type or L0 count data variable only, to give an abundance by each test group. 

In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# False = doesn't have an IR.  
# In controls these should all ne 'NA' and it will therefore give the number of cells with and without an IR.

#SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
 #                     groupby= "mdata:Sample_Tag_Name")

FILE = os.path.join(out_folder, "SampleAbundance_CLL_L0_ordinal.csv")

SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
                      groupby= VAR,
                      target_col="mdata:CLL_L0x10^9")

SampleAbundance_data_frame.to_csv(path_or_buf= FILE)

In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# False = doesn't have an IR.  
# In controls these should all ne 'NA' and it will therefore give the number of cells with and without an IR.

#SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
 #                     groupby= "mdata:Sample_Tag_Name")

FILE = os.path.join(out_folder, "SampleAbundance_CLL_L0_ordinal.csv")

SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
                      groupby= VAR,
                      target_col="mdata:CLL_L0x10^9"
                      )

SampleAbundance_data_frame.to_csv(path_or_buf= FILE)

In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# False = doesn't have an IR.  
# In controls these should all ne 'NA' and it will therefore give the number of cells with and without an IR.

#SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
 #                     groupby= "mdata:Sample_Tag_Name")

FILE = os.path.join(out_folder, "SampleAbundance_BCell_type.csv")

SampleAbundance_data_frame = ir.tl.group_abundance(mdata_airr, 
                      groupby= VAR,
                      target_col="mdata:BCell_type")

SampleAbundance_data_frame.to_csv(path_or_buf= FILE)

# Confirm removal of CLL clones
Can compare to output of BCR_multiBatch_identifyCLLclones.ipynb

In [ ]:
# Pre-processing - this part must be run for ALL analyses. 
# Requires the same parameters for metric and sequence as the scirpy.tl.define_clonotype_clusters fucntion. Note, the defaults for these two functions are NOT the same!
# BCR recommended parameters are in place here from https://scirpy.scverse.org/en/latest/generated/scirpy.pp.ir_dist.html#scirpy.pp.ir_dist.  

ir.pp.ir_dist(mdata_airr, metric="normalized_hamming", sequence="nt", histogram=True, cutoff=15) 




## Clonotypes

In [ ]:
# Define clonotypes

ir.tl.define_clonotypes (mdata_airr, key_added='clone_id') 

In [ ]:
# Clonotype clusters per sample or per group.
# Clone_id key adds clone_id and clone_id size to obs. This can be customised if it needs to be annotate per batch etc. 

ir.tl.define_clonotype_clusters(
    mdata_airr,
    sequence='nt',
    metric='normalized_hamming',
    receptor_arms='all',
    dual_ir='any',
    same_v_gene=True,
    same_j_gene=True,
    #within_group='mdata:Sample_Tag_Name',
    within_group=VAR,
    partitions="fastgreedy",
    key_added="clone_id",
)

In [ ]:
# Clonotype network generation.

ir.tl.clonotype_network(
    mdata_airr, sequence='nt', metric='normalized_hamming', min_cells=3, clonotype_key="clone_id"
)

In [ ]:
# Plot clonotype network.
# If using subsampled data the subsample will be the only ones coloured in here, but the context of the rest of the data is still displayed. More useful to run on the whole dataset. 
# If grey = removed upstream i.e. derived from multiplets or undetermined cells. 

FILE = os.path.join(out_folder, "ClonalNetwork.png")

fig, ax=plt.subplots(figsize=(15,15))

ir.pl.clonotype_network(
    mdata_airr,
    #color="mdata:Sample_Tag_Name",
    color=VAR,
    title='Clonotype Network BCR by CLL Burden',
    label_fontsize=9,
    base_size=20,
    ax=ax,
)
fig.savefig(
    fname= FILE,
    )

In [ ]:
# print(list(mdata_airr.obs.columns))
print([c for c in mdata_airr.obs.columns if any(k in c for k in ("cc", "identity", "clone"))])
# mdata_airr.obs.head()

In [ ]:
# Clonotype convergence - needs >1 sample to work

#ir.tl.clonotype_convergence(mdata_airr, key_coarse='mdata:GroupA_arm', key_fine='airr:clone_id')

ir.tl.clonotype_convergence(mdata_airr, key_coarse=VAR, key_fine='airr:clone_id')

## Repertoire Analysis

In [ ]:
# Clonal Expansion
# Adjust breakpoints parameter

#ir.tl.clonal_expansion(mdata_airr, target_col='clone_id', breakpoints=(1, 2),)

FILE = os.path.join(out_folder, "ClonalExpansion.txt")

ir.tl.clonal_expansion(
    mdata_airr, 
    target_col='clone_id', 
    breakpoints=(
        1, 
        2,
    )
)
expanded = np.sum(mdata_airr.obs["airr:clonal_expansion"] != "<= 1") / len(mdata_airr)

with open(FILE, "a") as f:
    print(f"{expanded: .2%} of cells belong to expanded (size >= 2) clonotype cluster in this dataset.", file=f)

In [ ]:
# Plot clonal expansion / group abundance

#ir.pl.group_abundance(
 #   mdata_airr, target_col="mdata:Sample_Tag_Name", groupby="airr:clonal_expansion", sort=["<= 1", "<= 2", "> 2"]
#)

FILE = os.path.join(out_folder, "ClonalExpansionBar.png")

fig, ax=plt.subplots()

ir.pl.group_abundance(
    mdata_airr, 
    target_col=VAR, 
    groupby="airr:clonal_expansion", 
    sort=["<= 1", "<= 2", "> 2"], 
    ax=ax
    )
fig.savefig(
    fname= FILE,
    bbox_inches='tight'
    )
#plt.savefig("PATH/outputs/Sample_Tag_Name_Results/ClonalExpansionBar.png", bbox_inches='6')
#ir.pl.group_abundance(mdata_airr, groupby="mdata:Sample_Tag_Name")

In [ ]:
# Summarise clonal expansion by group - use when groups defined - see documetation https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.tl.summarize_clonal_expansion.html#scirpy.tl.summarize_clonal_expansion
# This shows which samples have n number of cells with >2 BCRs that are the same. 

#ir.tl.summarize_clonal_expansion(mdata_airr, groupby="mdata:Sample_Tag_Name", breakpoints=(1,2,5,10,50))

ir.tl.summarize_clonal_expansion(mdata_airr, groupby=VAR, breakpoints=(1,2,5,10,50))

In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# False = doesn't have an IR.  
# In controls these should all ne 'NA' and it will therefore give the number of cells with and without an IR.

#ClonalExpansion_data_frame = ir.tl.summarize_clonal_expansion(mdata_airr, groupby="mdata:Sample_Tag_Name", breakpoints=(1,2,5,10,50))

FILE = os.path.join(out_folder, "ClonalExpansion.csv")

ClonalExpansion_data_frame = ir.tl.summarize_clonal_expansion(mdata_airr, groupby=VAR, breakpoints=(1,2,5,10,50))

ClonalExpansion_data_frame.to_csv(path_or_buf= FILE)

## Clonotype Diversity

In [ ]:
# Alpha diversity of clonotypes within a group - set when groups defined # https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.tl.alpha_diversity.html#scirpy.tl.alpha_diversity
# Alpha diversity is within-sample and measures the "richness and eveness" of the repertoire. https://academic.oup.com/bioinformatics/article/40/7/btae431/7702327

import skbio as sk

ir.tl.alpha_diversity(
    mdata_airr, 
    #groupby='mdata:Sample_Tag_Name', 
    groupby= VAR,
    target_col='clone_id', 
    metric='normalized_shannon_entropy'
    ) 


In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# False = doesn't have an IR.  
# In controls these should all ne 'NA' and it will therefore give the number of cells with and without an IR.

FILE = os.path.join(out_folder, "AlphaDiversity_abundance.csv")

AlphaDiversity_data_frame = ir.tl.alpha_diversity(
    mdata_airr, 
    #groupby='mdata:Sample_Tag_Name',
    groupby=VAR, 
    target_col='clone_id', 
    metric='normalized_shannon_entropy',
    inplace= False
    ) 

AlphaDiversity_data_frame.to_csv(path_or_buf= FILE)

In [ ]:
# Alpha diverity - needs groupby

fig, ax=plt.subplots(figsize=(8, 4))

FILE = os.path.join(out_folder, "AlphaDiversityBar.png")

ir.pl.alpha_diversity(
    mdata_airr,
    #groupby='mdata:Sample_Tag_Name',
    groupby=VAR,
    fig_kws={"dpi": 120}, figsize=[8, 4],
    ax=ax
)

fig.savefig(
    fname=FILE,
)

In [ ]:
# Repertorie overlap - requires a group e.g. samples, diagnosis etc.  - NOT working

ir.tl.repertoire_overlap (
    mdata_airr, 
    #groupby='mdata:Sample_Tag_Name',
    groupby=VAR,
    target_col='clone_id' 
)
#
   # target_col='clone_id', 
   # added_key='repertoire_overlap',
   # overlap_measure='jaccard',
    #overlap_threshold=None,
    #fraction=False,
    #inplace=True,
    #airr_mod='airr',
#)

In [ ]:
# Overlap between a pair of samples - needs an annotation with sample name and arm. Will tell me how similar sample repertoires are to one another. - NOT working. Even when after clonotype_imbalance below, it is still not returning a plot as expected. 

ir.tl.repertoire_overlap(
    mdata_airr,
    #groupby="mdata:Sample_Tag_Name",
    groupby=VAR,
    target_col="airr:clone_id",
    added_key="repertoire_overlap",
    overlap_measure="jaccard",
    #fraction="mdata:Sample_Tag_Name", 
    fraction=VAR, # <- use column name here
    airr_mod="airr",
)


In [ ]:
# Clonotype imbalance - clonotypes more enriched or depleted in a category. Must be run after repertoire overlap, which is not working so this will not work. 

#ir.tl.clonotype_imbalance(mdata_airr, replicate_col="mdata:Sample_Tag_Name", groupby="mdata:GroupA_arms", case_label="CLLPreRx")
ir.tl.clonotype_imbalance(mdata_airr, replicate_col=VAR, groupby="mdata:Sample_Tag_Name", case_label="CLLPreRx")

In [ ]:
ir.pl.clonotype_imbalance(mdata_airr, replicate_col=VAR, groupby="mdata:Sample_Tag_Name", case_label="CLLPreRx")

In [ ]:
#ir.tl.summarize_clonal_expansion(mdata_airr, groupby='mdata:Sample_Tag_Name', target_col='airr:clonal_expansion')
ir.tl.summarize_clonal_expansion(mdata_airr, groupby=VAR, target_col='airr:clonal_expansion')

## Clonotype Abundance ##

In [ ]:
# Group abundance
# Clone ID per sample.

FILE = os.path.join(out_folder, "ClonotypeAbundance.png")

fig, ax=plt.subplots(figsize=[8, 6])

ir.pl.group_abundance(
    mdata_airr,
    groupby='clone_id',
    #target_col='mdata:Sample_Tag_Name',
    target_col=VAR,
    max_cols=30,
    fig_kws={"dpi": 120}, figsize=[8, 6],
    ax=ax
)
fig.savefig(
    fname=FILE,
    bbox_inches='tight'
)

In [ ]:
# This is the number of cells the data is calculated on. In my data batch 5 had a BCR amplification error, and therefore the BCR counts are low compared to the number of cells.
# False = doesn't have an IR.  
# In controls these should all ne 'NA' and it will therefore give the number of cells with and without an IR.

FILE= os.path.join(out_folder, "CloneIDAbundance.csv")

CloneID_data_frame = ir.tl.group_abundance(mdata_airr, 
                      groupby= "clone_id",
                      #target_col="mdata:Sample_Tag_Name")
                      target_col=VAR)

CloneID_data_frame.to_csv(path_or_buf= FILE)

In [ ]:
# Return the nucleotide sequence of a specific clone_id - I can then find these in Cellismo (listed alphabetically) and delete the relevant clone.
# All clones have the same nucleotide sequence (this has been checked, but returns a large list for high frequence clones)
# In the AIRR file components 
# Junction is used because this includes the CDR3 plus the two flanking conserved codons. https://docs.airr-community.org/en/v1.2.1/datarep/rearrangements.html, AIRR uses the IMGT definitions. 
# print (mdata_airr.obs.head())

# mdata_airr.obs.query('airr:clone_id==82').head()

#clone_num = "34206"

#list(mdata_airr.obs[mdata_airr.obs["airr:clone_id"]==clone_num]["mdata:BCR_Heavy_CDR3_Junction_Nucleotide_Dominant"].values[0:1])
#list(mdata_airr.obs[mdata_airr.obs["airr:clone_id"]=="531"]["airr:junction_aa"])
# can add a range onto this to get specific range of clones .values[0:1])

In [ ]:
# Get the junction aa sequence of the clone:

#list(mdata_airr.obs[mdata_airr.obs["airr:clone_id"]=="82"]["mdata:BCR_Heavy_CDR3_Junction_Translation_Dominant"].values[0:1])

## Gene Usage

In [ ]:

#with ir.get.airr_context(mdata_airr, "v_call", chain=["VDJ_1"]) as m:
#    combinations = (
 #       m.obs.groupby(["mdata:Sample_Tag_Name", "VDJ_1_v_call"], observed=True)
  #      .size()
   #     .reset_index(name="n")
    #)

with ir.get.airr_context(mdata_airr, "v_call", chain=["VDJ_1"]) as m:
    combinations = (
        m.obs.groupby([VAR, "VDJ_1_v_call"], observed=True)
        .size()
        .reset_index(name="n")
    )    

In [ ]:
# Get VDJ_1 usage to define whether stereotypes over-represented. 

FILE= os.path.join(out_folder, "Vcall.png")

fig, ax=plt.subplots()

with ir.get.airr_context(mdata_airr, "v_call", chain="VDJ_1"):
    ax_Status = ir.pl.group_abundance(
        mdata_airr,
        groupby="VDJ_1_v_call",
        #target_col="mdata:Sample_Tag_Name",
        target_col=VAR,
        max_cols=20,
        figsize=(8, 4),
        ax=ax
    )
_= ax_Status.set_xticklabels(ax_Status.get_xticklabels(), rotation=50, ha="right")
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)
      

In [ ]:
#V_call CSV

FILE= os.path.join(out_folder, "V_call.csv")

with ir.get.airr_context(mdata_airr, "v_call", chain=["VDJ_1"]) as m:
    dataframe_vdj = (
        m.obs.groupby([VAR, "VDJ_1_v_call"], observed=True)
        .size()
    )

dataframe_vdj.to_csv(path_or_buf= FILE)

In [ ]:
# Get VDJ_1 usage of J to define whether stereotypes over-represented. 

FILE= os.path.join(out_folder, "Jcall.png")

fig, ax=plt.subplots()

with ir.get.airr_context(mdata_airr, "j_call", chain="VDJ_1"):
    ax_Status = ir.pl.group_abundance(
        mdata_airr,
        groupby="VDJ_1_j_call",
        #target_col="mdata:Sample_Tag_Name",
        target_col=VAR,
        max_cols=20,
        figsize=(4, 4),
        ax=ax,
    )
_= ax_Status.set_xticklabels(ax_Status.get_xticklabels(), rotation=50, ha="right")
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)
          

In [ ]:
#J_call CSV

FILE= os.path.join(out_folder, "J_call.csv")

with ir.get.airr_context(mdata_airr, "j_call", chain=["VDJ_1"]) as m:
    dataframe_vdj = (
        m.obs.groupby([VAR, "VDJ_1_j_call"], observed=True)
        .size()
    )

dataframe_vdj.to_csv(path_or_buf= FILE)

In [ ]:
# Get VDJ_1 usage of D to define whether stereotypes over-represented. 

FILE= os.path.join(out_folder, "Dcall.png")

fig, ax=plt.subplots()

with ir.get.airr_context(mdata_airr, "d_call", chain="VDJ_1"):
    ax_Status = ir.pl.group_abundance(
        mdata_airr,
        groupby="VDJ_1_d_call",
        #target_col="mdata:Sample_Tag_Name",
        target_col=VAR,
        max_cols=20,
        figsize=(4, 4),
        ax=ax
    )
_= ax_Status.set_xticklabels(ax_Status.get_xticklabels(), rotation=50, ha="right")
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)
     

In [ ]:
#D_call CSV

FILE= os.path.join(out_folder, "D_call.csv")

with ir.get.airr_context(mdata_airr, "d_call", chain=["VDJ_1"]) as m:
    dataframe_vdj = (
        m.obs.groupby([VAR, "VDJ_1_d_call"], observed=True)
        .size()
    )

dataframe_vdj.to_csv(path_or_buf= FILE)

In [ ]:
# Filter the stuff that makes VDJ come out as 'none' - need work

mask = ~mdata.obs["BCR_Heavy_V_gene_Dominant"].isin(["<unassigned>,"
"none"])
mdata_filtered = mdata[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in mdata_filtered.obs.columns:
    if pd.api.types.is_categorical_dtype(mdata_filtered.obs[col].dtype):
        mdata_filtered.obs[col] = mdata_filtered.obs[col].cat.remove_unused_categories()

mdata = mdata_filtered
mdata.shape

In [ ]:
# This is VDJ usage for the whole dataset! New mdata_airr objects will have to be made for per sample or per study arm. 

# VJ and VDJ onfo here, see reference https://pmc.ncbi.nlm.nih.gov/articles/PMC4654805/ for reasons why (mainly VJ calls more reliable than D calls), so both displayed. 

FILE= os.path.join(out_folder, "Ribbon.png")

fig, ax=plt.subplots(figsize=(12, 8))

ir.pl.vdj_usage(
    mdata_airr,
    vdj_cols=("VJ_1_v_call", "VJ_1_j_call", "VDJ_1_v_call", "VDJ_1_d_call", "VDJ_1_j_call"),
    full_combination=False,
    max_segments=None,
    max_labelled_segments=10,
    max_ribbons=30,
    fig_kws={"figsize": (12, 8)},
    ax=ax
)
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)

In [ ]:
# VDJ Spectratype - needs a groupby variable

#ir.tl.spectratype(mdata_airr, target_col='mdata:Sample_Tag_Name')

ir.tl.spectratype(mdata_airr, target_col=VAR)
                  
# groupby='IR_VJ_1_junction_nt', target_col='clone_id')

In [ ]:
# Spectratype - needs colour allocated by groups.
# color = <column to colur by e.g. sample, treatment group. 
# cdr3_col= <default junction_aa>, change to another airr column length attribute
# chain= <default 'VJ_1'>
# https://scirpy.scverse.org/en/latest/tutorials/tutorial_5k_bcr.html

# Spectratype will show the heterogentiy of VDJ sequences. In this usage, it can show samples with a probable clone not fitting into the normal distribution. It will also give a clue for the clone ID re. junction_aa attribute. 

## Spectratype 1 - junction_aa by Sample

FILE= os.path.join(out_folder, "Spectratype_aa.png")

fig, ax=plt.subplots(figsize=(8, 8))

with ir.get.airr_context(mdata_airr, "v_call"):
    ir.pl.spectratype(
        mdata_airr, 
        chain="VDJ_1", 
        cdr3_col="junction_aa", 
        viztype="bar", 
        #color="mdata:Sample_Tag_Name"
        color=VAR, 
        fig_kws={"dpi": 120}, 
        figsize=[8, 8],
        ax=ax,
    )
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)    

In [ ]:
## Spectratype 2 - junction_aa by VDJ_1_v_call for clones >=2
# not very informative when run for all VDJ calls in large datasets because too many categories = grey colour uniformly! Hence restricted to >=2.

FILE= os.path.join(out_folder, "Spectratype_clonalExpansion.png")

fig, ax=plt.subplots(figsize=(8, 8))

with ir.get.airr_context(mdata_airr, "v_call"):
    ir.pl.spectratype(
        mdata_airr[
            mdata_airr.obs["airr:clonal_expansion"].isin(["<= 2", "> 2"]),
            :,
        ], 
        chain="VDJ_1", 
        cdr3_col="junction_aa", 
        viztype="bar", 
        color="airr:clonal_expansion", 
        fig_kws={"dpi": 120}, 
        figsize=[8, 8],
        ax=ax
        )
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
) 

In [ ]:
# Spectratype for SUBSAMPLED data only, for clonal expansion >2, so will be skewed. Coloured by sample to see which ones it affects.

FILE= os.path.join(out_folder, "Spectratype_CloneMoreThan2.png")

fig, ax=plt.subplots(figsize=(8, 8))

with ir.get.airr_context(mdata_airr, "v_call"):
    ir.pl.spectratype(
        mdata_airr[
            mdata_airr.obs["airr:clonal_expansion"].isin(["> 2"]),
            :,
        ], 
        chain="VDJ_1", 
        cdr3_col="junction_aa", 
        viztype="bar", 
        #color="mdata:Sample_Tag_Name", 
        color=VAR,
        fig_kws={"dpi": 120}, 
        figsize=[8, 8],
        ax=ax
        )
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)

In [ ]:
# Visualise spectratype as a curve.

FILE= os.path.join(out_folder, "Spectratype_Curve.png")

fig, ax=plt.subplots(figsize=(6, 8))
                     
ir.pl.spectratype(
    mdata_airr,
    cdr3_col="junction_aa",
    chain="VDJ_1",
    #color="mdata:Sample_Tag_Name",
    color=VAR,
    viztype="curve",
    curve_layout="shifted",
    fig_kws={"figsize": [6, 8]},
    kde_kws={"kde_norm": False},
    ax=ax
)
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)

## Subsetting data 
see BCR_subsets_ipynb

Additional BCR data on junction length and amino acid sequences within the VDJ are shown in the scirpy tutorial, but are not relevant to this analysis at present. 
https://scirpy.scverse.org/en/latest/tutorials/tutorial_5k_bcr.html

## Somatic Hypermutation Analysis

If this is to be done, a germline aligment needs to be produced first. 

In [ ]:
mdata_airr

In [ ]:
## Get mutational load - BROKEN. But MUST run to get the violin plot. Returns a 'type error', likely a python issue. Runs enough to get info for the violin plot. 

# Data needs to be given in context of whether this is a sample with CLL clone still present, or if it has been removed; and the mutational status of the CLL clones if they are present (if known, if not know couldb be implied).



ir.tl.mutational_load(
                    mdata_airr, 
                    germline_key='germline_alignment',
                    regions="v")

#mutational_load_data_frame.to_csv(path_or_buf= FILE)

In [ ]:
#Mutational Load Dataframe

FILE = os.path.join(out_folder, "V_mutationalLoad.csv")

with ir.get.airr_context(mdata_airr, "v_mutation_freq", chain=["VDJ_1"]) as m:
    dataframe_mut = (
        m.obs.groupby([VAR, "VDJ_1_mutation_freq"], observed=True)
        .size()
    )

dataframe_mut.to_csv(path_or_buf= FILE)

#Dataframe= ir.get.airr(mdata_airr, "v_mutation_freq")
#Dataframe.to_csv(path_or_buf= FILE)


In [ ]:
FILE= os.path.join(out_folder, "Violin.png")

fig, ax=plt.subplots(figsize=(10, 6))

with ir.get.airr_context(mdata_airr, "v_mutation_freq"):
    sc.pl.violin(
        mdata_airr, ["VDJ_1_v_mutation_freq"], 
       # groupby="mdata:Sample_Tag_Name", 
        groupby=VAR,
        stripplot=False, 
        inner="box", 
        rotation=80, 
        legend=False,
        ax=ax
    )
fig.savefig(
    fname=FILE,
    bbox_inches="tight"
)

## Useful but not included in main pipeline

## Generic ##

Explore basic stats e.g. how many cells in a group.

In [ ]:
# Can sort into number of cells per group. May be useful to remove cell_ids with clonal VDJs. 

# Simple example - how many cells ina sample:

ir.tl.group_abundance(mdata_airr, 'mdata:Sample57')

# Other parameters: target_col='has_ir', airr_mod='airr', airr_key='airr', chain_idx_key='chain_indices', fraction=None, sort='count', all on https://scirpy.scverse.org/en/latest/generated/scirpy.tl.group_abundance.html#scirpy.tl.group_abundance

## Quality Control ##

In [ ]:
# Check expected return of QC. Run individually. 

mdata_airr.obs["airr:receptor_type"].head()
#mdata_airr.obs["airr:receptor_subtype"].head()
#mdata_airr.obs["airr:chain_pairing"].head()


## Combining multiple samples ##

https://scverse.org/scirpy/tags/v0.11.0/tutorials/tutorial_io.html#Combining-multiple-samples

This will be required alongside the grouping metadata when the samples are made into separate files according to sample tags. 

## Tool and plots ##

See documentation: https://scverse.org/scirpy/tags/v0.11.0/api.html

Each tool has a corresponding plot function.


## Useful code when errors arrise

In [ ]:
# Check keys
mdata_airr.obsp.keys()

## Plotting the outputs - appearance

In [ ]:
# Settings for graph appearance - see documentation if defaults are not ok - https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.pl.embedding.html#scirpy.pl.embedding

# ir.pl.embedding(adata, <basis>)

In [ ]:
# How many cells per group for multiple groups - useful for assessing how many cells I have data for per sample or per study condition. 

#scirpy.tl.group_abundance (adata) #see tools and plots documentation link above for syntax